# DeepLure AIE-CASE: Color-Invariant Saree Design Recognition
### End-to-End Deep Metric Learning for Textile Motif Identification & Verification

---

## 1. Approach Note (500 characters summary)
> **"Textile motif recognition is modeled via metric learning using ConvNeXt with learnable Generalized-Mean (GeM) pooling to isolate fine geometric weave patterns from global color. Training uses SubCenter-ArcFace loss (m=0.40, s=30.0) with P-K balanced sampling, paired with aggressive hue/saturation perturbations and adversarial color disentanglement via Gradient Reversal (GRL) to penalize palette predictability. Evaluation uses L2-cosine similarity for CMC/mAP retrieval and ROC/EER verification."**

---

In [ ]:
# Install dependencies (if running in Kaggle / Colab)
!pip install -q timm>=0.9.7 albumentations>=1.3.0 pyyaml>=6.0

In [ ]:
import os
import math
import time
import random
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.autograd import Function
import torchvision.transforms as T
import timm
from sklearn.metrics import roc_curve, auc, precision_recall_curve

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Active Compute Device: {device}")

## 2. Color-Invariant Data Augmentation Pipeline
Standard CNNs and ViTs heavily latch onto color histograms (e.g. bright red vs emerald green). To force the network to learn geometric weaves, borders, butis, and pallu motifs, we employ an aggressive pipeline of **circular hue cycling, random channel shuffling, posterization, solarization, and random grayscale conversion**.

In [ ]:
class RandomGrayscaleOrInvert(torch.nn.Module):
    def __init__(self, p_gray: float = 0.35, p_invert: float = 0.1):
        super().__init__()
        self.p_gray = p_gray
        self.p_invert = p_invert

    def forward(self, img: Image.Image) -> Image.Image:
        if random.random() < self.p_invert:
            return ImageOps.invert(img.convert("RGB"))
        elif random.random() < self.p_gray:
            return ImageOps.grayscale(img).convert("RGB")
        return img

class RandomPosterizeSolarize(torch.nn.Module):
    def __init__(self, p_posterize: float = 0.2, p_solarize: float = 0.2):
        super().__init__()
        self.p_posterize = p_posterize
        self.p_solarize = p_solarize

    def forward(self, img: Image.Image) -> Image.Image:
        if random.random() < self.p_posterize:
            img = ImageOps.posterize(img, random.randint(2, 4))
        if random.random() < self.p_solarize:
            img = ImageOps.solarize(img, random.randint(64, 192))
        return img

class RandomChannelShuffle(torch.nn.Module):
    def __init__(self, p: float = 0.25):
        super().__init__()
        self.p = p

    def forward(self, img: Image.Image) -> Image.Image:
        if random.random() < self.p:
            channels = list(img.split())
            if len(channels) == 3:
                random.shuffle(channels)
                return Image.merge("RGB", channels)
        return img

def get_train_transforms(image_size=(256, 256)):
    return T.Compose([
        T.RandomResizedCrop(size=image_size, scale=(0.7, 1.0), ratio=(0.8, 1.25)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        RandomChannelShuffle(p=0.25),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.8, hue=0.5),
        RandomPosterizeSolarize(p_posterize=0.2, p_solarize=0.2),
        RandomGrayscaleOrInvert(p_gray=0.35, p_invert=0.1),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.2, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
    ])

def get_val_transforms(image_size=(256, 256)):
    return T.Compose([
        T.Resize(image_size),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

## 3. Dataset & P-K Balanced Batch Sampler

In [ ]:
class SareeDataset(Dataset):
    def __init__(self, image_paths: List[str], labels: List[int], transform=None, return_color_label=False):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.return_color_label = return_color_label
        self.unique_labels = sorted(list(set(labels)))
        self.label_to_idx = {lbl: idx for idx, lbl in enumerate(self.unique_labels)}
        self.num_classes = len(self.unique_labels)
        self.indexed_labels = [self.label_to_idx[lbl] for lbl in labels]
        self.class_to_indices = {}
        for idx, lbl in enumerate(self.indexed_labels):
            self.class_to_indices.setdefault(lbl, []).append(idx)

    def __len__(self):
        return len(self.image_paths)

    def _extract_color_cluster(self, img: Image.Image) -> int:
        small_img = img.resize((16, 16)).convert("HSV")
        arr = np.array(small_img)
        avg_hue = np.mean(arr[:, :, 0])
        avg_sat = np.mean(arr[:, :, 1])
        if avg_sat < 35: return 0
        return min(1 + int((avg_hue / 256.0) * 7), 7)

    def __getitem__(self, idx: int):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        color_label = self._extract_color_cluster(img) if self.return_color_label else -1
        label = self.indexed_labels[idx]
        img_tensor = self.transform(img) if self.transform else T.ToTensor()(img)
        if self.return_color_label:
            return img_tensor, label, color_label, path
        return img_tensor, label, path

class PKBalancedSampler(Sampler):
    def __init__(self, dataset: SareeDataset, p: int = 8, k: int = 4, length: int = 100):
        super().__init__(dataset)
        self.dataset = dataset
        self.p = min(p, dataset.num_classes)
        self.k = k
        self.length = length
        self.available_classes = [c for c, idxs in dataset.class_to_indices.items() if len(idxs) > 0]

    def __len__(self):
        return self.length

    def __iter__(self):
        for _ in range(self.length):
            batch = []
            selected = random.sample(self.available_classes, self.p)
            for cls in selected:
                indices = self.dataset.class_to_indices[cls]
                if len(indices) >= self.k:
                    batch.extend(random.sample(indices, self.k))
                else:
                    batch.extend([random.choice(indices) for _ in range(self.k)])
            random.shuffle(batch)
            yield batch

## 4. Synthetic Multi-Colorway Textile Dataset Generator
Allows instant zero-dependency execution and testing with multi-palette geometric weaves (zari checks, polka butis, stripes, paisleys, mandalas).

In [ ]:
def create_synthetic_dataset(output_dir="./data/synthetic", num_designs=20, colorways_per_design=6):
    os.makedirs(output_dir, exist_ok=True)
    palettes = [
        ((180, 20, 50), (218, 165, 32), (10, 25, 80)),
        ((20, 140, 60), (192, 192, 192), (110, 20, 30)),
        ((30, 60, 200), (245, 245, 245), (220, 180, 30)),
        ((128, 0, 128), (218, 165, 32), (25, 25, 25)),
        ((0, 128, 128), (240, 110, 90), (245, 240, 220)),
        ((210, 30, 150), (30, 200, 210), (60, 60, 60)),
    ]
    train_p, train_l, gal_p, gal_l, qry_p, qry_l = [], [], [], [], [], []
    
    for d in range(num_designs):
        d_dir = Path(output_dir) / f"design_{d:03d}"
        d_dir.mkdir(parents=True, exist_ok=True)
        imgs = []
        for c in range(colorways_per_design):
            pal = palettes[(d + c) % len(palettes)]
            img = Image.new("RGB", (256, 256), color=pal[0])
            draw = ImageDraw.Draw(img)
            ptype = d % 4
            if ptype == 0:
                for x in range(0, 256, 32):
                    draw.line([(x, 0), (x+128, 256)], fill=pal[1], width=2)
                    draw.line([(x, 256), (x+128, 0)], fill=pal[1], width=2)
                draw.rectangle([(0, 220), (256, 256)], fill=pal[2])
            elif ptype == 1:
                for x in range(20, 256, 40):
                    for y in range(20, 220, 40):
                        draw.ellipse([(x-8, y-8), (x+8, y+8)], fill=pal[1])
                draw.rectangle([(0, 220), (256, 256)], fill=pal[2])
            elif ptype == 2:
                for x in range(0, 256, 24):
                    draw.line([(x, 0), (x, 220)], fill=pal[1], width=3)
                draw.rectangle([(0, 220), (256, 256)], fill=pal[2])
            else:
                for x in [64, 192]:
                    for y in [64, 160]:
                        draw.ellipse([(x-25, y-25), (x+25, y+25)], outline=pal[1], width=3)
                draw.rectangle([(0, 220), (256, 256)], fill=pal[2])
            
            fpath = str(d_dir / f"color_{c:02d}.jpg")
            img.save(fpath, quality=95)
            imgs.append(fpath)
            
        # Split: 3 train, 2 gallery, 1 query
        for p in imgs[:3]: train_p.append(p); train_l.append(d)
        for p in imgs[3:5]: gal_p.append(p); gal_l.append(d)
        for p in imgs[5:]: qry_p.append(p); qry_l.append(d)
        
    return train_p, train_l, gal_p, gal_l, qry_p, qry_l

train_p, train_l, gal_p, gal_l, qry_p, qry_l = create_synthetic_dataset()
train_ds = SareeDataset(train_p, train_l, transform=get_train_transforms(), return_color_label=True)
gallery_ds = SareeDataset(gal_p, gal_l, transform=get_val_transforms())
query_ds = SareeDataset(qry_p, qry_l, transform=get_val_transforms())
print(f"Dataset Ready: Train={len(train_ds)}, Gallery={len(gallery_ds)}, Query={len(query_ds)}")

## 5. Model Architecture: ConvNeXt + GeM + SubCenter ArcFace + GRL Color Disentanglement

In [ ]:
class GeM(nn.Module):
    def __init__(self, p: float = 3.0, eps: float = 1e-6, trainable: bool = True):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p, requires_grad=trainable)
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class SubCenterArcMarginProduct(nn.Module):
    def __init__(self, in_features: int, out_features: int, sub_centers: int = 3, s: float = 30.0, m: float = 0.40):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.sub_centers = sub_centers
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features * sub_centers, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_embeddings: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        norm_feat = F.normalize(input_embeddings)
        norm_weight = F.normalize(self.weight)
        cosine = F.linear(norm_feat, norm_weight)
        B = cosine.size(0)
        cosine = cosine.view(B, self.out_features, self.sub_centers)
        cosine, _ = torch.max(cosine, dim=2)
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=input_embeddings.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.s

class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class ColorInvariantSareeNet(nn.Module):
    def __init__(self, backbone_name="convnext_nano.in12k_ft_in1k", num_classes=20, embedding_dim=256, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        self.pooling = GeM(p=3.0, trainable=True)
        self.neck = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.1),
            nn.Linear(in_features, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim),
        )
        self.head = SubCenterArcMarginProduct(in_features=embedding_dim, out_features=num_classes, sub_centers=3)
        self.color_head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 8),
        )

    def extract_features(self, x):
        feats = self.backbone.forward_features(x)
        pooled = self.pooling(feats).flatten(1)
        emb = self.neck(pooled)
        return F.normalize(emb, p=2, dim=1)

    def forward(self, x, labels=None):
        feats = self.backbone.forward_features(x)
        pooled = self.pooling(feats).flatten(1)
        raw_emb = self.neck(pooled)
        norm_emb = F.normalize(raw_emb, p=2, dim=1)
        out = {"embedding": norm_emb}
        if labels is not None:
            out["logits"] = self.head(raw_emb, labels)
        if self.training:
            rev_emb = GradReverse.apply(raw_emb, 0.2)
            out["color_logits"] = self.color_head(rev_emb)
        return out

model = ColorInvariantSareeNet(num_classes=train_ds.num_classes, pretrained=False).to(device)

## 6. Training with Supervised Metric Loss & Adversarial Color Penalty

In [ ]:
sampler = PKBalancedSampler(train_ds, p=6, k=3, length=15)
train_loader = DataLoader(train_ds, batch_sampler=sampler)
gallery_loader = DataLoader(gallery_ds, batch_size=16, shuffle=False)
query_loader = DataLoader(query_ds, batch_size=16, shuffle=False)

criterion_margin = nn.CrossEntropyLoss(label_smoothing=0.05)
criterion_color = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

print("[*] Training for 5 demo epochs...")
for epoch in range(5):
    model.train()
    total_loss = 0.0
    for imgs, labels, color_labels, _ in train_loader:
        imgs, labels, color_labels = imgs.to(device), labels.to(device), color_labels.to(device)
        optimizer.zero_grad()
        out = model(imgs, labels=labels)
        l_margin = criterion_margin(out["logits"], labels)
        l_color = criterion_color(out["color_logits"], color_labels)
        loss = l_margin + 0.15 * l_color
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5 | Total Loss: {total_loss/len(train_loader):.4f}")

## 7. Evaluation: Identification (mAP / CMC) & Verification (ROC-AUC / EER)

In [ ]:
@torch.no_grad()
def extract_feats(loader):
    model.eval()
    embs, lbls, paths = [], [], []
    for imgs, labels, p in loader:
        embs.append(model.extract_features(imgs.to(device)).cpu())
        lbls.append(labels.numpy())
        paths.extend(p)
    return torch.cat(embs, dim=0), np.concatenate(lbls), paths

gal_feats, gal_labels, gal_paths = extract_feats(gallery_loader)
qry_feats, qry_labels, qry_paths = extract_feats(query_loader)

# Cosine distance & similarity
sim_mat = torch.mm(qry_feats, gal_feats.t()).numpy()
dist_mat = 1.0 - sim_mat

# Identification Metrics
cmc_1 = np.mean([gal_labels[np.argsort(dist_mat[i])[0]] == qry_labels[i] for i in range(len(qry_labels))]) * 100

# Verification Metrics
sims_flat = sim_mat.flatten()
gt_flat = (qry_labels[:, None] == gal_labels[None, :]).flatten().astype(int)
fpr, tpr, _ = roc_curve(gt_flat, sims_flat)
roc_auc = auc(fpr, tpr) * 100
eer = float(fpr[np.nanargmin(np.abs(fpr - (1.0 - tpr)))]) * 100

print(f"\n{'='*50}")
print(f"EVALUATION RESULTS:")
print(f" • Rank-1 Identification Accuracy : {cmc_1:.2f}%")
print(f" • Verification ROC-AUC           : {roc_auc:.2f}%")
print(f" • Equal Error Rate (EER)         : {eer:.2f}%")
print(f"{'='*50}")

## 8. Efficiency & Complexity Profiling (Deliverable 4)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
model_size_mb = (total_params * 4) / (1024 * 1024)

# CPU Latency Benchmark
dummy = torch.randn(1, 3, 256, 256).to(device)
with torch.no_grad():
    for _ in range(10): _ = model.extract_features(dummy)
    start = time.perf_counter()
    for _ in range(30): _ = model.extract_features(dummy)
    lat_ms = ((time.perf_counter() - start) / 30) * 1000.0

print(f"\nEFFICIENCY REPORT:")
print(f" • Parameter Count         : {total_params/1e6:.2f} M")
print(f" • Model Memory Footprint  : {model_size_mb:.2f} MB")
print(f" • Embedding Size          : 256 dimensions (1,024 bytes/saree)")
print(f" • 100k Gallery RAM Index  : {(100000 * 256 * 4)/(1024*1024):.2f} MB")
print(f" • Inference Latency       : {lat_ms:.2f} ms ({1000.0/lat_ms:.1f} FPS)")